In [ ]:
!pip install sentencepiece transformers tqdm
!git clone https://github.com/ggarciabaez/VLA.git
!pip install --upgrade torch torchvision

In [2]:
import os, shutil
from google.colab import drive
from tqdm import tqdm
drive.mount('/content/drive')
src = "/content/drive/MyDrive/VLA/mt50/"
dst = "/content/data/"
os.makedirs(dst, exist_ok=True)
files = os.listdir(src)
for file in tqdm(files):
  if file.endswith(".npz"):
    shutil.copy(src+file, dst)
shutil.copy(src+"task_prompts.json", dst)
shutil.copytree("/content/VLA/model", "/content/model")
shutil.rmtree("/content/VLA")

Mounted at /content/drive


'/content/data'

In [1]:
import json
import math
import os
import random
import time
from collections import OrderedDict
from contextlib import nullcontext
from pathlib import Path
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.amp import GradScaler
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm
from transformers import SiglipTokenizer

from model.vla import VLA, print_model_counts
from model.utils import VLAConfig

In [2]:
# =========================
# Colab Config (edit here)
# =========================
DATA_DIR = "/content/data/"
CHECKPOINT_DIR = "/content/data/checkpoints"
PROMPTS_JSON = "task_prompts.json"
EPISODE_GLOB = "ep*.npz"
MAX_EPISODE_FILES = None  # int or None

SEED = 42
VAL_SPLIT = 0.1
BATCH_SIZE = 256
NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = False

EPOCHS = 10
LEARNING_RATE = 3e-4
BACKBONE_LR_SCALE = 0.1
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2
GRAD_CLIP_NORM = 1.0
LOG_EVERY_STEPS = 100
RESUME = False
COMPILE_MODEL = False

# Model config
MODEL_KWARGS = dict(
    n_trainable=4,
    d_model=768,
    n_heads=6,
    n_layers=8,
    latent_size=64,
    action_heads=6,
    action_layers=4,
    chunk_size=32,
    flow_steps=10,
    dropout=0.1,
)

In [3]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_task_name(x) -> str:
    if isinstance(x, bytes):
        return x.decode("utf-8")
    return str(x)

In [4]:
class MT50Dataset(Dataset):
    def __init__(self, data_dir="/content/data", cfg: VLAConfig = VLAConfig(), n_tasks=50, n_steps=200):
        self.cfg = cfg
        self.data = glob.glob(os.path.join(data_dir, "ep*.npz"))
        self.n_steps, self.total_tasks, self.n_tasks = n_steps, n_tasks*len(self.data), n_tasks
        with np.load(os.path.join(data_dir, "norm_stats.npz")) as f:
            self.mean, self.std = f["action_mean"], f["action_std"]

        # a brave choice of preallocating, not using zeros
        self.images = np.empty((n_steps, self.total_tasks, 3, cfg.img_size, cfg.img_size), dtype=np.uint8)
        self.actions = np.empty((n_steps, self.total_tasks, cfg.action_dim), dtype=np.float32)
        self.states = np.empty((n_steps, self.total_tasks, cfg.state_dim), dtype=np.float32)
        self.prompts = None  # this is handled differently
        self.task_names = []
        self.chunk_indices = None  # we don't know the chunk size yet

        offset = 0
        for filename in tqdm(self.data):
            with np.load(filename) as f:
                n = f["images"].shape[1]
                self.images[:, offset:offset+n] = f["images"]
                self.actions[:, offset:offset+n] = f["actions"]
                self.states[:,  offset:offset+n] = f["states"]
                offset += n
                if not self.task_names:
                    self.task_names = f["task_names"].tolist()
                if self.chunk_indices is None:
                    self.chunk_indices = f["chunk_indices"].astype(np.uint8)  # (n_steps, 32), same for all files

        with open(os.path.join(data_dir, "task_prompts.json")) as f:
            task_keys, task_vals = list(zip(*json.load(f).items()))
        tokens = SiglipTokenizer.from_pretrained(cfg.siglip_model_id).encode(
            np.array(task_vals).flatten().tolist(),
            padding="max_length", return_tensors="pt"
        ).reshape(-1, 3, 64)

        self.prompts = {task_keys[i]: tokens[i] for i in range(len(task_keys))}
        self.actions -= self.mean
        self.actions /= self.std
        self.prompt_tensor = torch.stack([
          self.prompts[name] for name in self.task_names
        ])

        self.total_gb = (self.images.size*self.images.itemsize
            + self.actions.size*self.actions.itemsize
            + self.states.size*self.states.itemsize
            + self.chunk_indices.size*self.chunk_indices.itemsize
            + sum(p.numel() * p.element_size() for p in self.prompts.values()))/1e+9

        print(f"Loaded {self.total_tasks*n_steps} entries, {self.total_gb}GB of memory.")
        self.images = torch.from_numpy(self.images)        # uint8
        self.actions = torch.from_numpy(self.actions)      # float32
        self.states = torch.from_numpy(self.states)
        self.chunk_indices = torch.from_numpy(self.chunk_indices)

    def __len__(self):
        return self.n_steps*self.total_tasks

    def __getitem__(self, idx):
        task, step = divmod(idx, self.n_steps)
        act_chunk = self.chunk_indices[step]
        return (
            self.images[step, task], # uint8, (B, 3, 224, 224)
            self.actions[act_chunk.tolist(), task], # float32 norm, (B, 32, 4)
            self.states[step, task],  # (B, 39)
            self.prompt_tensor[task, np.random.randint(0, 3)],  # (B, 64)
        )

class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)

In [5]:
def save_checkpoint(path: str, model: nn.Module, optimizer, scaler, epoch: int, step: int, best_loss: float):
    torch.save(
        {
            "epoch": epoch,
            "step": step,
            "best_loss": best_loss,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "config": model.cfg,
        },
        path,
    )

def load_checkpoint(path: str, model: nn.Module, optimizer, scaler):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    print(f"Resumed from {path} (epoch={ckpt['epoch']}, step={ckpt['step']})")
    return int(ckpt["epoch"]), int(ckpt["step"]), float(ckpt["best_loss"])


def build_model_cfg(stats: dict[str, torch.Tensor]) -> VLAConfig:
    cfg = VLAConfig(**MODEL_KWARGS)
    cfg.action_mean = stats["action_mean"].tolist()
    cfg.action_std = stats["action_std"].tolist()
    return cfg


def build_optimizer(model: VLA, lr: float, backbone_lr_scale: float, weight_decay: float):
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "backbone" in name:
            backbone_params.append(p)
        else:
            head_params.append(p)

    return torch.optim.AdamW(
        [
            {"params": head_params, "lr": lr},
            {"params": backbone_params, "lr": lr * backbone_lr_scale},
        ],
        weight_decay=weight_decay,
        fused=torch.cuda.is_available()
    )

In [8]:
tmp_cfg = VLAConfig(**MODEL_KWARGS)
dataset = MT50Dataset(
        data_dir=DATA_DIR,
        cfg=tmp_cfg,
)

Dataset: 20 episodes, 200,000 samples


In [ ]:
def train(dataset):
    seed_everything(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    cfg = build_model_cfg({"action_mean": dataset.mean, "action_std": dataset.std})

    # --- 1. Task-Level Splits ---
    # Instead of splitting random steps, we split entire tasks.
    all_tasks = list(range(dataset.total_tasks))
    random.shuffle(all_tasks)

    val_len = max(1, int(len(all_tasks) * VAL_SPLIT))
    val_tasks = all_tasks[:val_len]
    train_tasks = all_tasks[val_len:]
    print(f"Train tasks: {len(train_tasks)} | Val tasks: {len(val_tasks)}")

    model = VLA(cfg, device=device).to(device)
    if COMPILE_MODEL:
        model = torch.compile(model)

    total, trainable = print_model_counts(model)
    print(f"Total params: {total:,} | Trainable: {trainable:,} | Frozen: {total - trainable:,}")

    optimizer = build_optimizer(
        model, lr=LEARNING_RATE, backbone_lr_scale=BACKBONE_LR_SCALE, weight_decay=WEIGHT_DECAY
    )

    # --- 2. Calculate Steps ---
    batches_per_epoch = math.ceil(len(train_tasks) / BATCH_SIZE)
    chunks_per_task = math.ceil(dataset.n_steps / cfg.seq_len)
    total_steps = EPOCHS * batches_per_epoch * chunks_per_task
    warmup_steps = WARMUP_EPOCHS * batches_per_epoch * chunks_per_task

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    bf16_ok = device.type == "cuda" and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if bf16_ok else torch.float16
    amp_enabled = device.type == "cuda"
    scaler = GradScaler(device=device.type, enabled=(amp_enabled and amp_dtype == torch.float16))

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    latest_path = os.path.join(CHECKPOINT_DIR, "latest.pt")
    best_path = os.path.join(CHECKPOINT_DIR, "best.pt")

    start_epoch, global_step, best_val = 0, 0, float("inf")
    if RESUME and os.path.exists(latest_path):
        start_epoch, global_step, best_val = load_checkpoint(latest_path, model, optimizer, scaler)
        start_epoch += 1

    # --- 3. Custom Chronological Training Loop ---
    for epoch in tqdm(range(start_epoch, EPOCHS)):
        model.train()
        epoch_train_meter = AverageMeter()
        log_meter = AverageMeter()
        t0 = time.perf_counter()

        random.shuffle(train_tasks)

        for i in range(0, len(train_tasks), BATCH_SIZE):
            task_batch = train_tasks[i : i + BATCH_SIZE]
            current_b = len(task_batch)

            # Reset memory at the very beginning of the task batch
            model.memory.reset(current_b, device, amp_dtype)

            # Pre-fetch text prompts for this task batch
            tok = torch.stack([
                dataset.prompt_tensor[idx%dataset.n_tasks, np.random.randint(0, 3)] for idx in task_batch
            ]).to(device, non_blocking=True)

            # Step through time chronologically
            for t in range(0, dataset.n_steps, cfg.seq_len):
                current_seq_len = min(cfg.seq_len, dataset.n_steps - t)

                # Slicing datasets (in CPU RAM)
                # Images: (W, B, C, H, W) -> transpose to (B, W, C, H, W)
                img = dataset.images[t : t + current_seq_len, task_batch]
                img = img.transpose(0, 1).to(device, non_blocking=True)

                # States: (W, B, D) -> transpose to (B, W, D)
                state = dataset.states[t : t + current_seq_len, task_batch]
                state = state.transpose(0, 1).to(device, non_blocking=True)

                # Actions: Complex indexing to get (B, W, 32, action_dim)
                acts = dataset.actions[:, task_batch] # (200, B, action_dim)
                c_idx = dataset.chunk_indices[t : t + current_seq_len].long() # (W, 32)
                chunk = acts[c_idx] # (W, 32, B, action_dim)
                chunk = chunk.permute(2, 0, 1, 3).to(device, non_blocking=True) # (B, W, 32, action_dim)

                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, dtype=amp_dtype):
                    loss = model.loss_seq(img, tok, state, chunk)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

                # CRITICAL: Detach memory so gradients don't flow across sequence boundaries
                model.memory.detach()

                epoch_train_meter.update(loss.item(), current_b * current_seq_len)
                log_meter.update(loss.item(), current_b * current_seq_len)
                global_step += 1

                if global_step % LOG_EVERY_STEPS == 0:
                    elapsed = max(time.perf_counter() - t0, 1e-6)
                    sps = LOG_EVERY_STEPS * (BATCH_SIZE * cfg.seq_len) / elapsed
                    lr_now = optimizer.param_groups[0]["lr"]
                    print(
                        f"epoch {epoch:>3d} step {global_step:>7d} "
                        f"loss {log_meter.avg:.4f} lr {lr_now:.2e} {sps:,.0f} samples/s"
                    )
                    log_meter.reset()
                    t0 = time.perf_counter()

        # --- 4. Validation Loop ---
        model.eval()
        val_meter = AverageMeter()
        with torch.no_grad():
            for i in range(0, len(val_tasks), BATCH_SIZE * 2):
                task_batch = val_tasks[i : i + BATCH_SIZE * 2]
                current_b = len(task_batch)
                model.memory.reset(current_b, device, amp_dtype)

                tok = torch.stack([
                    dataset.prompt_tensor[idx%dataset.n_tasks, np.random.randint(0, 3)] for idx in task_batch
                ]).to(device, non_blocking=True)

                for t in range(0, dataset.n_steps, cfg.seq_len):
                    current_seq_len = min(cfg.seq_len, dataset.n_steps - t)

                    img = dataset.images[t : t + current_seq_len, task_batch].transpose(0, 1).to(device, non_blocking=True)
                    state = dataset.states[t : t + current_seq_len, task_batch].transpose(0, 1).to(device, non_blocking=True)

                    acts = dataset.actions[:, task_batch]
                    c_idx = dataset.chunk_indices[t : t + current_seq_len].long()
                    chunk = acts[c_idx].permute(2, 0, 1, 3).to(device, non_blocking=True)

                    autocast_ctx = torch.autocast(device_type=device.type, dtype=amp_dtype) if amp_enabled else nullcontext()
                    with autocast_ctx:
                        val_loss = model.loss_seq(img, tok, state, chunk)

                    model.memory.detach() # Detach in eval just to clean up buffers
                    val_meter.update(val_loss.item(), current_b * current_seq_len)

        print(f"epoch {epoch:>3d} train_loss {epoch_train_meter.avg:.4f} val_loss {val_meter.avg:.4f}")

        save_checkpoint(latest_path, model, optimizer, scaler, epoch, global_step, best_val)
        if val_meter.avg < best_val:
            best_val = val_meter.avg
            save_checkpoint(best_path, model, optimizer, scaler, epoch, global_step, best_val)
            print(f"new best: {best_val:.4f} -> {best_path}")

    print("Training complete.")

In [ ]:
train(dataset)

In [ ]:
os.makedirs("/content/drive/MyDrive/VLA/mt50/model_v2", exist_ok=True)
shutil.copytree(CHECKPOINT_DIR, "/content/drive/MyDrive/VLA/mt50/model_v2", dirs_exist_ok=True)
from google.colab import runtime
import time
time.sleep(300)  # wait 5m for everything to finish up
runtime.unassign()